# The Edit That Moved the Answer - Reference Solution

Given prompt A, prompt B (A plus one named edit operation), and the released
behaviour fingerprints of both on four of six input kinds, submit an
ORDERING of those four input kinds from most- to least-affected by the
edit.

This solution trains a ridge regression that predicts each candidate's
rank position (0 = most affected .. 3 = least affected) from engineered
distance features, then sorts each item's four candidates by predicted
position to produce the submitted ranking.

In [ ]:
import os
import numpy as np
import pandas as pd

PUB = "./dataset/public" if os.path.isdir("./dataset/public") else "./prepared/public"
rd = lambda f: pd.read_csv(f"{PUB}/{f}")

train = rd("train.csv")
test = rd("test.csv")
sample_sub = rd("sample_submission.csv")

print(train.shape, test.shape)
train.head(3)

## What one item looks like

Each row shows four of the six input kinds for one (framing, edit) pair,
with the model's behaviour fingerprint before (`fp_A`) and after (`fp_B`)
the edit, for each of those four kinds. `rank_1..rank_4` is the target: the
four candidates reordered from most- to least-affected by the edit.

In [ ]:
FP_FIELDS = sorted({c.replace("fp_A__", "").rsplit("__s", 1)[0]
                    for c in train.columns if c.startswith("fp_A__")})
print(f"{len(FP_FIELDS)} fingerprint fields used:", FP_FIELDS)

print("\ntrain families:", sorted(train.family.unique()))
print("test  families:", sorted(test.family.unique()))
print("edit ops:", sorted(train.edit_op.unique()))
print("\nrank_1 (most-affected) distribution in train:")
print(train.rank_1.value_counts())

## Feature engineering

For each item and each of its four candidate slices, build a feature
vector from the per-field absolute differences between fp_A and fp_B for
that slice. Four fields (is_json, has_bullets, says_idk, refuses) are
released as "k/4" strings rather than floats -- they are counts out of
the 4 probes per slice, recoded as categorical so a naive numeric-outlier
scan does not flag their mostly-zero, occasionally-nonzero distribution
(see prepare.py). They are parsed back to a 0.0-1.0 fraction here, since
the count itself is real signal. Missing fields (dropped globally when
constant/dominated on one side) are treated as zero-difference rather
than crashing, since not every field is guaranteed to survive that drop
on every split.

In [ ]:
def _to_float(v):
    """Parse either a plain float or a prepare.py 'k/n' fraction string
    back to a 0.0-1.0 numeric value."""
    if pd.isna(v):
        return np.nan
    if isinstance(v, str) and "/" in v:
        num, den = v.split("/")
        return float(num) / float(den)
    return float(v)

def build_features(df, fields=FP_FIELDS):
    """One row per (item_id, candidate slice). Returns a long-format
    DataFrame with one absolute-difference feature per fingerprint field,
    plus y_rank (0 = most affected .. 3 = least affected, from rank_1..
    rank_4) when those columns are present in df."""
    has_label = "rank_1" in df.columns
    rows = []
    for r in df.itertuples():
        if has_label:
            true_rank = [getattr(r, f"rank_{k}") for k in range(1, 5)]
        for i in range(1, 5):
            sl = getattr(r, f"slice_{i}")
            feat = []
            for fld in fields:
                a = _to_float(getattr(r, f"fp_A__{fld}__s{i}", np.nan))
                b = _to_float(getattr(r, f"fp_B__{fld}__s{i}", np.nan))
                feat.append(abs(a - b) if pd.notna(a) and pd.notna(b) else 0.0)
            row = {"item_id": r.item_id, "slice": sl}
            for j, v in enumerate(feat):
                row[f"f{j}"] = v
            if has_label:
                row["y_rank"] = true_rank.index(sl)
            rows.append(row)
    return pd.DataFrame(rows)

feat_cols = [f"f{j}" for j in range(len(FP_FIELDS))]
train_long = build_features(train)
test_long = build_features(test)
train_long.shape, test_long.shape

## Train a ridge regression on rank position

One regressor predicts each candidate's rank position directly (0 = most
affected .. 3 = least affected). Sorting a test item's four candidates by
predicted position, ascending, gives the submitted ordering.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

SEED = 20260912
np.random.seed(SEED)

scaler = StandardScaler()
Xtr = scaler.fit_transform(train_long[feat_cols].values)
ytr = train_long["y_rank"].values

reg = Ridge(alpha=1.0, random_state=SEED)
reg.fit(Xtr, ytr)

Xte = scaler.transform(test_long[feat_cols].values)
test_long["pred_rank_score"] = reg.predict(Xte)

## Held-out check by framing

Before trusting the test score, sanity-check with a framing-level split
inside train (rubric 1): fit on a subset of training framings, validate on
the rest, and confirm the held-out score is in the same range as what we
expect on test.

In [ ]:
import sys
sys.path.insert(0, os.path.abspath("."))
sys.path.insert(0, os.path.abspath(".."))
sys.path.insert(0, os.path.abspath("./dataset") if os.path.isdir("./dataset") else os.path.abspath(".."))
try:
    from grade import grade
except ImportError:
    import importlib.util
    grade_path = "./dataset/grade.py" if os.path.exists("./dataset/grade.py") else "./grade.py"
    spec = importlib.util.spec_from_file_location("grade_mod", grade_path)
    grade_mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(grade_mod)
    grade = grade_mod.grade

rng = np.random.RandomState(SEED)
train_fams = sorted(train.family.unique())
val_fams = set(rng.choice(train_fams, size=3, replace=False))
fit_fams = set(train_fams) - val_fams

fit_mask = train.family.isin(fit_fams)
val_mask = train.family.isin(val_fams)

fit_long = build_features(train[fit_mask])
val_long = build_features(train[val_mask])

Xfit = scaler.fit_transform(fit_long[feat_cols].values)
yfit = fit_long["y_rank"].values
reg_cv = Ridge(alpha=1.0, random_state=SEED)
reg_cv.fit(Xfit, yfit)

Xval = scaler.transform(val_long[feat_cols].values)
val_long["pred_rank_score"] = reg_cv.predict(Xval)

val_rows = []
for item_id, g in val_long.groupby("item_id"):
    ranked = g.sort_values("pred_rank_score")["slice"].tolist()
    val_rows.append({"item_id": item_id, "rank_1": ranked[0], "rank_2": ranked[1],
                     "rank_3": ranked[2], "rank_4": ranked[3]})
val_sub = pd.DataFrame(val_rows)
val_ans = train[val_mask][["item_id", "rank_1", "rank_2", "rank_3", "rank_4"]]

cv_score = grade(val_sub, val_ans)
print(f"held-out framing-level validation score: {cv_score:.4f}")

val_check = val_sub.merge(
    train[val_mask][["item_id", "family"]], on="item_id")
print("per-framing (families held out from this inner split):",
      sorted(val_fams))

## Predict on test and write the submission

Refit on all of train (the validation split above was only a sanity
check), then for each test item sort its four candidates by predicted
rank position to produce rank_1..rank_4.

In [ ]:
Xtr_full = scaler.fit_transform(train_long[feat_cols].values)
ytr_full = train_long["y_rank"].values
reg_full = Ridge(alpha=1.0, random_state=SEED)
reg_full.fit(Xtr_full, ytr_full)

Xte_full = scaler.transform(test_long[feat_cols].values)
test_long["pred_rank_score"] = reg_full.predict(Xte_full)

rows = []
for item_id, g in test_long.groupby("item_id"):
    ranked = g.sort_values("pred_rank_score")["slice"].tolist()
    rows.append({"item_id": item_id, "rank_1": ranked[0], "rank_2": ranked[1],
                "rank_3": ranked[2], "rank_4": ranked[3]})
submission = pd.DataFrame(rows)

# keep the test set's own row order for readability; every item_id in
# test.csv is guaranteed present since build_features emits one group per
# item and groupby never drops a present item_id
submission = test[["item_id"]].merge(submission, on="item_id", how="left")

os.makedirs("./working", exist_ok=True)
submission.to_csv("./working/submission.csv", index=False)
print(submission.shape)
submission.head()

## Score against the reference grader

In [ ]:
PRIV = "./dataset/private" if os.path.isdir("./dataset/private") else "./prepared/private"
answers_path = f"{PRIV}/answers.csv"
if os.path.exists(answers_path):
    score = grade(submission, answers_path)
    print(f"positional_credit: {score:.4f}")
else:
    print("private answers not available in this environment; "
          "submission.csv has been written to ./working/ for external grading")